# 04 — MIDAS Regression
Mixed Data Sampling: forecast daily silver returns using monthly macro variables.

Key idea: instead of forward-filling monthly data to daily (lossy), MIDAS uses a
polynomial weighting scheme (Beta or Almon lag) to directly regress high-frequency
on low-frequency data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120

## 1. Load data

In [ ]:
# Daily target
train = pd.read_csv('../data/processed/train.csv', index_col=0, parse_dates=True)
test  = pd.read_csv('../data/processed/test.csv',  index_col=0, parse_dates=True)

# Monthly macro (raw — not forward-filled)
macro = pd.read_csv('../data/raw/monthly_macro.csv', index_col=0, parse_dates=True)

TARGET     = 'silver_return'
MACRO_VARS = ['cpi', 'fed_funds', 'ind_prod', 'real_rates']
MACRO_VARS = [c for c in MACRO_VARS if c in macro.columns]

print('Monthly macro vars available:', MACRO_VARS)

## 2. U-MIDAS (Unrestricted MIDAS)
Simplest form: stack lagged monthly values as regressors. Good baseline before imposing Beta/Almon weights.


In [ ]:
def build_umidas_features(daily_target: pd.Series, monthly: pd.DataFrame,
                           macro_vars: list, n_lags: int = 3) -> pd.DataFrame:
    """
    For each macro variable, stack its last n_lags monthly observations
    available as of each daily date.
    """
    rows = []
    for date in daily_target.index:
        row = {'date': date, TARGET: daily_target.loc[date]}
        past_macro = monthly[monthly.index <= date].tail(n_lags)
        for var in macro_vars:
            if var not in past_macro.columns:
                continue
            vals = past_macro[var].values
            for i, v in enumerate(reversed(vals)):
                row[f'{var}_lag{i+1}'] = v
        rows.append(row)
    df = pd.DataFrame(rows).set_index('date')
    return df.dropna()

print('Building U-MIDAS features...')
umidas_train = build_umidas_features(train[TARGET].dropna(), macro, MACRO_VARS)
umidas_test  = build_umidas_features(test[TARGET].dropna(),  macro, MACRO_VARS)
print(f'Train: {umidas_train.shape}, Test: {umidas_test.shape}')

## 3. Fit U-MIDAS with OLS

In [ ]:
import statsmodels.api as sm

feat_cols = [c for c in umidas_train.columns if c != TARGET]

X_tr = sm.add_constant(umidas_train[feat_cols])
y_tr = umidas_train[TARGET]
X_te = sm.add_constant(umidas_test[feat_cols]).reindex(columns=X_tr.columns, fill_value=0)
y_te = umidas_test[TARGET]

ols = sm.OLS(y_tr, X_tr).fit()
print(ols.summary())

## 4. U-MIDAS forecast & evaluation

In [ ]:
preds_umidas = ols.predict(X_te)
actuals      = y_te.values

from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(actuals, preds_umidas))
mae  = mean_absolute_error(actuals, preds_umidas)
da   = np.mean(np.sign(actuals) == np.sign(preds_umidas))

nonzero = np.abs(actuals) > 1e-6
mape = np.mean(np.abs((actuals[nonzero] - preds_umidas[nonzero]) / actuals[nonzero])) * 100

correct = np.sign(actuals) == np.sign(preds_umidas)
wda = np.sum(np.abs(actuals) * correct) / np.sum(np.abs(actuals))

print(f'U-MIDAS  RMSE={rmse:.6f}  MAE={mae:.6f}  MAPE={mape:.2f}%  DA={da:.3f}  W-DA={wda:.3f}')


## 5. MIDAS with Beta polynomial weights (via midaspy)
Beta weights enforce a smooth, hump-shaped decay over lags — more parsimonious than U-MIDAS.


In [ ]:
try:
    from midaspy import WeightMethod, MidasAdl

    # Use CPI as the low-frequency variable
    if 'cpi' in macro.columns:
        y_daily   = train[TARGET].dropna()
        x_monthly = macro['cpi'].dropna()

        # Number of high-freq obs per low-freq period (daily/monthly ≈ 22)
        m = MidasAdl(y_daily, x_monthly, poly=WeightMethod.Beta,
                     lags=3, num_high_freq=22)
        m.fit()
        print(m.summary())
except ImportError:
    print('midaspy not installed or API differs — using U-MIDAS results above.')
except Exception as e:
    print(f'midaspy error: {e}')
    print('Continuing with U-MIDAS results.')

## 6. Save MIDAS metrics

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_te.index, actuals,       label='Actual',   lw=1)
ax.plot(y_te.index, preds_umidas,  label='U-MIDAS',  lw=1, alpha=0.8)
ax.set_title('U-MIDAS 1-step Forecast — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/midas_forecast_plot.png', dpi=150, bbox_inches='tight')
plt.show()

pd.DataFrame([{'model': 'U-MIDAS', 'rmse': rmse, 'mae': mae, 'mape': mape,
               'dir_acc': da, 'wda': wda}]
            ).to_csv('../data/processed/metrics_midas.csv', index=False)
print('Metrics saved.')

pd.DataFrame({'actual': actuals, 'predicted': preds_umidas}).to_csv(
    '../data/processed/preds_umidas.csv', index=False)
print('Prediction arrays saved.')
